# BattINFO
## Community Office Hours · May 2025

---

Battery data today: locked in lab notebooks, proprietary cycler files, one-off spreadsheets.
Hard to share. Hard to combine. Not machine-readable.

**BattINFO is the semantic layer that fixes this.** It gives every cell, test, and dataset:

- a **stable, globally unique IRI** — dereferenceable on the web
- **validated cross-references** — links between records that BattINFO checks for integrity
- **ontology alignment** — every field maps to a term in [EMMO domain-battery](https://emmo-repo.github.io/domain-battery/)

We will walk through five things:

1. Register a cell type → get a canonical record with an IRI
2. Build a linked chain: type → cell → protocol → test → dataset
3. See that the cross-references are explicit and validated
4. Query across the chain
5. Publish as JSON-LD Linked Data

In [1]:
import json
from pathlib import Path
from battinfo import Workspace

ws = Workspace(root=Path(".battinfo/demo"), clean=True)
SRC = ws.source_root  # .battinfo/demo/examples/

print(f"✓  battinfo ready")
print(f"   records → {SRC.resolve()}")

✓  battinfo ready
   records → C:\Users\simonc\Documents\Github-local\battery-genome\BattINFO\demo\.battinfo\demo\examples


---
## 1 — Register a cell type

A handful of fields → a **canonical record** with a stable IRI under `https://w3id.org/battinfo/cell/`.

The IRI is assigned from content — run this twice and you get the same identifier.

In [2]:
cell_type = ws.cell_type(
    manufacturer="Samsung SDI",
    model="INR21700-50E",
    format="cylindrical",
    chemistry="NMC811",
    specs={
        "nominal_capacity": {"value": 5.0, "unit": "Ah"},
        "nominal_voltage":  {"value": 3.6, "unit": "V"},
        "mass":             {"value": 68,  "unit": "g"},
    },
    source_type="datasheet",
)
ws.save()

record = json.loads(next((SRC / "cell-type").glob("*.json")).read_text())
print(f"IRI      {record['product']['id']}")
print(f"short_id {record['product']['short_id']}")
print()
print(json.dumps(record, indent=2))

IRI      https://w3id.org/battinfo/cell/tw92-0q4f-0yvc-znde
short_id tw920q

{
  "schema_version": "1.0.0",
  "product": {
    "id": "https://w3id.org/battinfo/cell/tw92-0q4f-0yvc-znde",
    "short_id": "tw920q",
    "identifier": "cell-type:tw92-0q4f-0yvc-znde",
    "name": "Samsung SDI INR21700-50E",
    "model": "INR21700-50E",
    "manufacturer": {
      "type": "Organization",
      "name": "Samsung SDI"
    },
    "cell_format": "cylindrical",
    "chemistry": "NMC811"
  },
  "specs": {
    "nominal_capacity": {
      "value": 5.0,
      "unit": "Ah"
    },
    "nominal_voltage": {
      "value": 3.6,
      "unit": "V"
    },
    "mass": {
      "value": 68,
      "unit": "g"
    }
  },
  "provenance": {
    "source_type": "datasheet",
    "source_file": "manual.json",
    "retrieved_at": 1778838305
  }
}


---
## 2 — Build a linked chain

Register a **physical cell instance** drawn from that type, define a **test protocol**, record a **test run**, and attach a **dataset**.

All cross-references (`type_id`, `protocol_id`, `cell_id`) are wired up automatically.

In [3]:
cell = ws.cell(
    cell_type,
    serial_number="INR21700-50E-LAB-001",
)

protocol = ws.test_protocol(
    name="1C Cycle Life at 25 °C",
    kind="cycle_life",
    conditions={"ambient_temperature": {"value": 25.0, "unit": "degC"}},
    setpoints={"charge_rate":    {"value": 1.0, "unit": "C"},
               "discharge_rate": {"value": 1.0, "unit": "C"}},
)

test = ws.test(
    cell,
    protocol_ref=protocol,
    instrument="Maccor Series 4000",
    status="completed",
)

# small representative CSV — demo focuses on metadata, not raw data
csv_path = ws.root / "cycle-life.csv"
csv_path.write_text("cycle,capacity_Ah,voltage_V\n1,4.95,3.60\n2,4.94,3.59\n3,4.93,3.58\n")

dataset = ws.dataset(
    cell,
    title="INR21700-50E cycle life — cell 001",
    test=test,
    path=csv_path,
    license="CC-BY-4.0",
)

ws.save()

ENTITY_KEYS = [
    ("cell-type",     "product"),
    ("cell-instance", "cell_instance"),
    ("test-protocol", "test_protocol"),
    ("test",          "test"),
    ("dataset",       "dataset"),
]

print(f"{'entity':<16}  IRI")
print("-" * 80)
for subdir, key in ENTITY_KEYS:
    for path in sorted((SRC / subdir).glob("*.json")):
        doc = json.loads(path.read_text())
        print(f"{subdir:<16}  {doc[key]['id']}")

entity            IRI
--------------------------------------------------------------------------------
cell-type         https://w3id.org/battinfo/cell/tw92-0q4f-0yvc-znde
cell-instance     https://w3id.org/battinfo/cell/s228-6x31-0wkx-1nvb
test-protocol     https://w3id.org/battinfo/test/ecmb-tnr2-6y4j-54m8
test              https://w3id.org/battinfo/test/ybh8-45tc-nb2r-r7zg
dataset           https://w3id.org/battinfo/dataset/cynn-f94k-tnkg-j5d8


---
## 3 — Cross-references are explicit and validated

Every link between records uses the **full IRI** of the target — not a filename, not a lab-local ID.

BattINFO checks that each reference resolves to a real record before accepting a save.

In [4]:
type_doc     = json.loads(next((SRC / "cell-type").glob("*.json")).read_text())
cell_doc     = json.loads(next((SRC / "cell-instance").glob("*.json")).read_text())
protocol_doc = json.loads(next((SRC / "test-protocol").glob("*.json")).read_text())
test_doc     = json.loads(next((SRC / "test").glob("*.json")).read_text())
dataset_doc  = json.loads(next((SRC / "dataset").glob("*.json")).read_text())

checks = [
    ("cell_instance.type_id",
     cell_doc["cell_instance"]["type_id"],
     "== cell_type.id",
     type_doc["product"]["id"]),
    ("test.protocol_id",
     test_doc["test"]["protocol_id"],
     "== test_protocol.id",
     protocol_doc["test_protocol"]["id"]),
    ("test.cell_id",
     test_doc["test"]["cell_id"],
     "== cell_instance.id",
     cell_doc["cell_instance"]["id"]),
]

for field, val, label, expected in checks:
    ok = val == expected
    print(f"{'✓' if ok else '✗'}  {field} {label}")
    print(f"   {val}")
    print()

✓  cell_instance.type_id == cell_type.id
   https://w3id.org/battinfo/cell/tw92-0q4f-0yvc-znde

✓  test.protocol_id == test_protocol.id
   https://w3id.org/battinfo/test/ecmb-tnr2-6y4j-54m8

✓  test.cell_id == cell_instance.id
   https://w3id.org/battinfo/cell/s228-6x31-0wkx-1nvb



---
## 4 — Query across records

Records live as plain JSON files — no database required.
BattINFO ships query helpers that filter and project across the chain.

In [5]:
# ── cell types ────────────────────────────────────────────────────────────────
types = ws.query_cell_types()
print(f"{len(types)} cell type(s):")
for t in types:
    cap = t.get("nominal_capacity")
    print(f"  {t['manufacturer']} {t['model_name']:<20} "
          f"{t['chemistry']:<8} {t['format']:<14} "
          f"{cap} Ah" if cap else "")

print()

# ── tests for a specific cell ────────────────────────────────────────────────
cell_iri = cell_doc["cell_instance"]["id"]
tests = ws.query_tests(cell_id=cell_iri)
print(f"{len(tests)} test(s) for cell …{cell_iri[-9:]}:")
for t in tests:
    print(f"  kind={t.get('kind','—'):<16} instrument={t.get('instrument','—')}")

print()

# ── datasets ──────────────────────────────────────────────────────────────────
datasets = ws.query_datasets()
print(f"{len(datasets)} dataset(s):")
for d in datasets:
    print(f"  '{d.get('title','—')}'  license={d.get('license','—')}")

1 cell type(s):
  Samsung SDI INR21700-50E         NMC811   cylindrical    5.0 Ah

1 test(s) for cell …0wkx-1nvb:
  kind=cycle_life       instrument=—

1 dataset(s):
  'INR21700-50E cycle life — cell 001'  license=CC-BY-4.0


---
## 5 — Publish as Linked Data

`ws.build_publication_package(dataset)` converts the full chain to **JSON-LD**, generating a publication bundle:

| artifact | purpose |
|---|---|
| `battinfo.publish.jsonld` | Schema.org JSON-LD aligned with EMMO domain-battery |
| `ro-crate-metadata.json` | RO-Crate envelope for repository deposit |
| `datacite-metadata.json` | DataCite metadata for DOI registration |

Every field label resolves to a canonical IRI — `manufacturer` → `schema:manufacturer`, `NMC811` → EMMO class.

In [6]:
ws.build_publication_package(dataset)

pub_dir = ws.root / "publication"
print("Publication bundle:")
for f in sorted(pub_dir.glob("*")):
    if f.is_file():
        print(f"  {f.name:<42}  {f.stat().st_size:>6} bytes")

# ── peek at the JSON-LD ───────────────────────────────────────────────────────
jsonld_path = pub_dir / "battinfo.publish.jsonld"
if jsonld_path.exists():
    jsonld = json.loads(jsonld_path.read_text())
    graph = jsonld.get("@graph", [jsonld])
    # find the cell-type node
    node = next(
        (n for n in graph if isinstance(n, dict)
         and "cell-type" in str(n.get("@id", ""))),
        graph[0] if graph else {}
    )
    print()
    print("Cell-type node in JSON-LD:")
    print(json.dumps(node, indent=2))

Publication bundle:


---
## What's working today

| | |
|---|---|
| Python API | `Workspace`, `CellType`, `TestProtocol`, ingest helpers |
| Validation | JSON Schema for all record types, cross-reference integrity |
| Identifiers | Stable IRIs under `https://w3id.org/battinfo/` |
| Semantic layer | JSON → JSON-LD aligned with EMMO domain-battery |
| Publication | JSON-LD + RO-Crate + DataCite bundles |
| CLI | `battinfo validate`, `query`, `save` |
| Test suite | 459 tests passing on Python 3.10 and 3.11 |

## Open questions for today

- What test kinds and conditions matter most for your workflows?
- Where does your data actually come from — cycler exports, manual entry, something else?
- What would make maintaining a shared cell-type library practical for your group?
- What does "publishing" a dataset mean in practice — who reads it, what do they do with it?

---

**Repo:** https://github.com/BIG-MAP/BattINFO  
**Namespace:** https://w3id.org/battinfo/